Building an on premise healthcare Data processing platform

In [2]:
#importing required libraries
from pyspark.sql import SparkSession

# building spark session
spark = (SparkSession.builder.appName("CareBridge on-premise Platform").master("local[*]").getOrCreate())


print("Spark  version:", spark.version)

Spark  version: 4.2.0


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CareBridge On-Premise Platform")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 4.2.0


Define the file path

In [4]:
data_path = "../data/hospital_visits.csv"

print("Reading data from:", data_path)

Reading data from: ../data/hospital_visits.csv


using the Spark to read the CSV dataset 

In [5]:
df = (spark.read
    .option("header", "True")
    .option("inferSchema", "True")
    .csv(data_path)
)

In [6]:
df.show(10)

+--------+----------+----------------+---------+-------------+----------+----------------+------------+---------------+
|visit_id|patient_id|      department|doctor_id|   visit_type|visit_date|consultation_fee|visit_status|         branch|
+--------+----------+----------------+---------+-------------+----------+----------------+------------+---------------+
|   V0001|     P0007|General Medicine|     D024|    Follow-up|2026-02-05|           12000|   Completed|          Lekki|
|   V0002|     P0174|General Medicine|     D024|Routine Check|2026-04-19|           12000|   Cancelled|          Lekki|
|   V0003|     P0008|General Medicine|     D003| Consultation|2026-06-04|           12000|   Cancelled|          Lekki|
|   V0004|     P0144|General Medicine|     D007|Routine Check|2026-05-20|           12000|     Pending|Victoria Island|
|   V0005|     P0057|     Dermatology|     D015|Routine Check|2026-02-10|           18000|   Completed|          Ikeja|
|   V0006|     P0088|     Dermatology|  

In [ ]:
df.printSchema() #print the schema of the dataframe alongside the data types of each column 

root
 |-- visit_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- department: string (nullable = true)
 |-- doctor_id: string (nullable = true)
 |-- visit_type: string (nullable = true)
 |-- visit_date: date (nullable = true)
 |-- consultation_fee: integer (nullable = true)
 |-- visit_status: string (nullable = true)
 |-- branch: string (nullable = true)



In [8]:
print("Number of records in the dataframe: ", df.count())

Number of records in the dataframe:  500


In [10]:
from pyspark.sql.functions import col, count, sum, avg

In [ ]:
department_visits = ( 
    df
    .groupBy("department")
    .agg(
        count("*").alias("department_count"),
    )
    .orderBy(
        col("department_count").desc()
    )
) # This code groups the DataFrame df by the "department" column, counts the total number for each department, and orders the results in descending order based on the total visits.

department_visits.show()

+----------------+----------------+
|      department|department_count|
+----------------+----------------+
|     Dermatology|             113|
|      Cardiology|             104|
|General Medicine|             104|
|     Orthopedics|              91|
|      Pediatrics|              88|
+----------------+----------------+



In [13]:
#  visit by branch
branch_visits = (
    df
    .groupBy("branch")
    .agg(
        count("*").alias("total_visits")
    )
    .orderBy(
        col("total_visits").desc()
    )
) # This code groups the DataFrame df by the "branch" column, counts the total number of visits for each branch, and orders the results in descending order based on the total visits.

branch_visits.show()

+---------------+------------+
|         branch|total_visits|
+---------------+------------+
|Victoria Island|         186|
|          Ikeja|         177|
|          Lekki|         137|
+---------------+------------+



In [14]:
#completed  visits only
completed_visits = df.filter(
    col("visit_status") == "Completed"
) # This code filters the DataFrame df to include only the rows where the "visit_status" column has the value "Completed". The resulting DataFrame is stored in the variable completed_visits.

completed_visits.show()


+--------+----------+----------------+---------+-------------+----------+----------------+------------+---------------+
|visit_id|patient_id|      department|doctor_id|   visit_type|visit_date|consultation_fee|visit_status|         branch|
+--------+----------+----------------+---------+-------------+----------+----------------+------------+---------------+
|   V0001|     P0007|General Medicine|     D024|    Follow-up|2026-02-05|           12000|   Completed|          Lekki|
|   V0005|     P0057|     Dermatology|     D015|Routine Check|2026-02-10|           18000|   Completed|          Ikeja|
|   V0006|     P0088|     Dermatology|     D009| Consultation|2026-01-27|           18000|   Completed|          Lekki|
|   V0008|     P0118|General Medicine|     D018| Consultation|2026-05-22|           12000|   Completed|          Ikeja|
|   V0009|     P0161|      Pediatrics|     D020|    Follow-up|2026-01-18|           15000|   Completed|Victoria Island|
|   V0010|     P0170|General Medicine|  

In [ ]:
# revenue by department
# only completed visits are considered for revenue calculation
# this is an exellent little business rule to ensure that only completed visits are considered for revenue calculation, as it reflects the actual revenue generated by the hospital.
# A cancelled or no-show visit would not generate any revenue, so including them in the revenue calculation would give an inaccurate picture of the hospital's financial performance.


department_revenue = (
    df
    .filter(
        col("visit_status") == "Completed"
    )
    .groupBy("department")
    .agg(
        sum("consultation_fee").alias("total_revenue")
    )
    .orderBy(
        col("total_revenue").desc()
    )
) # This code filters the DataFrame df to include only the rows where the "visit_status" column has the value "Completed", groups the filtered DataFrame by the "department" column, calculates the total revenue for each department by summing the "consultation_fee" column, and orders the results in descending order based on the total revenue.

department_revenue.show()


+----------------+-------------+
|      department|total_revenue|
+----------------+-------------+
|      Cardiology|      1625000|
|     Dermatology|      1422000|
|     Orthopedics|      1320000|
|      Pediatrics|       930000|
|General Medicine|       756000|
+----------------+-------------+



In [16]:
# writing the result back  to local storage using pandas

department_revenue_pd = department_revenue.toPandas()

department_revenue_pd.to_csv(
    "../output/department_revenue.csv",
    index=False
) # This code converts the department_revenue DataFrame to a Pandas DataFrame and saves it as a CSV file named department_revenue.csv in the output folder, without including the index column in the CSV file.

print("Department revenue saved successfully.")

c:\Users\agunb\Documents\GitHub\carebridge_onprem_data_platform_ETL_pipeline\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:298: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
c:\Users\agunb\Documents\GitHub\carebridge_onprem_data_platform_ETL_pipeline\venv\Lib\site-packages\pyspark\sql\pandas\conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


Department revenue saved successfully.


In [19]:
#Postgresql database connection and writing the result to a table
#Before running the code
# Ensure that the PostgreSQL server is running and accessible locally.
#create a database named "carebridge" in PostgreSQL if it doesn't already exist.
#cretae a table named "department_revenue" in the "carebridge" database with the following schema: 
#Theraeafter you can run the code below to write (load) the department revenue data to the PostgreSQL table.

revenue_rows = department_revenue.collect() # Collect the rows of the department_revenue DataFrame

revenue_data = [
    (row["department"], row["total_revenue"]) # Creates a tuple for each row containing the department and total revenue
    for row in revenue_rows # Iterate through each row in the collected rows
]


In [25]:
#connect to the postgressql database and write the data to the table

import psycopg2

connection = psycopg2.connect(
    host="localhost",
    port="5432",
    database="carebridge_bd",
    user="postgres",
    password="Beauty123#"
)

cursor = connection.cursor()

cursor.execute("TRUNCATE TABLE department_revenue;")

for department, total_revenue in revenue_data:

    cursor.execute(
        """
        INSERT INTO department_revenue
        (department, total_revenue)
        VALUES (%s, %s)
        """,
        (department, total_revenue)
    )


connection.commit()

cursor.close()
connection.close() 
# Here, the data is loaded into the local PostgreSQL database. The connection to the database is closed after the data is inserted.

print("Data loaded into local PostgreSQL successfully.")

Data loaded into local PostgreSQL successfully.


In [24]:
spark.stop() # Stop the Spark session to free up resources